# ATML PA3 Task 2 — Run All
Select **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all** and approve the normal Google Drive mount.

This notebook runs the full 196-token, 4,500-update protocol. Re-running automatically recovers compatible checkpoints and reuses validated caches. Results are saved under `/content/drive/MyDrive/ATML_PA3_Task2/`.

**Verification:** 29 CPU tests, real official-data selection, and pretrained SigLIP/Qwen CPU integration passed. T4 memory, FP16 CUDA recovery, full training, and five-hour feasibility are **not yet verified**. The launcher measures and hard-gates these before training. No browser execution was used.

The default five-hour **session budget** includes a 15-minute safety reserve; it is not a claim that the entire experiment finishes in five hours. Run All in the next session resumes a safe stop.

Implementation commit: `11e166e110cf35e3f08e5ce087b69554f3b1634d`. Task 1 is untouched. No report prose is generated.


In [1]:
# Immutable implementation: notebook edits do not silently change the training source.
from pathlib import Path
import os
import subprocess
import sys
from google.colab import drive

SOURCE_COMMIT = "11e166e110cf35e3f08e5ce087b69554f3b1634d"
REPOSITORY = "https://github.com/Mujtaba7734/Advanced-Topics-in-Machine-Learning.git"
drive.mount("/content/drive")
source = Path("/content/atml_task2_source") / SOURCE_COMMIT
source.mkdir(parents=True, exist_ok=True)
def git(*args):
    return subprocess.check_output(["git", "-C", str(source), *args], text=True).strip()
if not (source / ".git").exists():
    git("init")
    git("remote", "add", "origin", REPOSITORY)
try:
    checked_out = git("rev-parse", "HEAD")
except subprocess.CalledProcessError:
    checked_out = None
if checked_out != SOURCE_COMMIT:
    git("fetch", "--depth", "1", "origin", SOURCE_COMMIT)
    git("checkout", "--detach", SOURCE_COMMIT)
assert git("rev-parse", "HEAD") == SOURCE_COMMIT
if git("status", "--porcelain"):
    raise RuntimeError("Pinned source has local edits; refusing to run a modified experiment.")
workdir = source / "Assignment-3"
env = dict(os.environ, TOKENIZERS_PARALLELISM="false", OMP_NUM_THREADS="2", HF_HUB_DISABLE_TELEMETRY="1")
print("Task 2 source:", SOURCE_COMMIT, flush=True)
subprocess.run([sys.executable, "-m", "task2.bootstrap"], cwd=workdir, env=env, check=True)
# Real CUDA gates run first. Failure stops here; no silent protocol fallback.
# On reconnect, Run All automatically resumes the latest compatible verified checkpoint.
subprocess.run([sys.executable, "-u", "-m", "task2.run"], cwd=workdir, env=env, check=True)


Mounted at /content/drive
Task 2 source: 11e166e110cf35e3f08e5ce087b69554f3b1634d


CompletedProcess(args=['/usr/bin/python3', '-u', '-m', 'task2.run'], returncode=0)